# Notebook for computing hashes, buckets and similarity values for the disk scheme using a hybrid approach

Utilizes the disk scheme

Incorporates:
* Hashing of trajectories using disk scheme
* Bucketing of hashes made from disk scheme
* Similarity computation between trajectories within buckets.
    * Both for DTW and Frechet
* Analysis of the produced bucket system

Produces:
* JSON file containing buckets
* Similarity values for trajectories within buckets


# Hybrid approach

In [ ]:
import os
import sys

def find_project_root(target_folder="masteroppgave"):
    """Find the absolute path of a folder by searching upward."""
    currentdir = os.path.abspath("__file__")  # Get absolute script path
    while True:
        if os.path.basename(currentdir) == target_folder:
            return currentdir  # Found the target folder
        parentdir = os.path.dirname(currentdir)
        if parentdir == currentdir:  # Stop at filesystem root
            return None
        currentdir = parentdir  # Move one level up

# Example usage
project_root = find_project_root("masteroppgave")

if project_root:
    sys.path.append(project_root)
    print(f"Project root found: {project_root}")
else:
    raise RuntimeError("Could not find 'masteroppgave' directory")

from computation.similarity import *
from utils.helpers.bucket_evaluation import *
import json
import pandas as pd


## Setup

In [ ]:
CITY = "rome" # "rome" or "porto"
MEASURE = "dtw" # "dtw" or "frechet"
BUCKETING_METHOD = "loose" # Bucketing method to use

DIAMETER_BUCKETING = 1.6  # Diameter to use for bucketing
LAYERS_BUCKETING = 2  # Number of layers to use for bucketing
DISKS_BUCKETING = 50  # Number of disks to use for bucketing

DIAMETER_COMPRESSION = 1.5  # Diameter to use for trajectory compression
LAYERS_COMPRESSION = 5  # Number of layers to use for trajectory compression
DISKS_COMPRESSION = 40  # Number of disks to use for trajectory compression

DATA_SIZE = 100  # Example dataset sizes

## Retrieving true similarity values

In [ ]:
file_path = f"../../../results_true/similarity_values/{CITY}/{MEASURE}/{CITY}-{MEASURE}-{DATA_SIZE}.csv"

# Read CSV, telling pandas to take the first column as the row labels:
true_sim_matrix_df = pd.read_csv(file_path, index_col=0)

# Function to convert values to float if possible
def convert_to_float(value):
    try:
        return float(value)
    except ValueError:
        return value

# Apply the function to each cell in the DataFrame
true_sim_matrix_df = true_sim_matrix_df.map(convert_to_float)
true_sim_matrix_df = (true_sim_matrix_df + true_sim_matrix_df.T)


## Generate hashes with hybrid disk scheme, bucket system and similarity values for given city and measure

In [ ]:
hashed_similarities, bucket_system = generate_disk_hash_similarity_with_bucketing_hybrid(
    city=CITY, diameter_bucketing=DIAMETER_BUCKETING, layers_bucketing=LAYERS_BUCKETING, disks_bucketing=DISKS_BUCKETING, diameter_compression=DIAMETER_COMPRESSION, layers_compression=LAYERS_COMPRESSION, disks_compression=DISKS_COMPRESSION, measure=MEASURE, size=DATA_SIZE, bucketing_method=BUCKETING_METHOD
)

In [ ]:
hashed_similarities.head(10)

# Bucket analysis

## Bucket stats

In [ ]:
# Evaluate the bucketing system
bucket_evaluation = evaluate_bucket_system(bucket_system)
bucket_evaluation.head(20)

In [ ]:
# THRESHOLD = 5
import numpy as np # type: ignore
THRESHOLDS = np.arange(1, 6.0, 1)  # Generates [0.5, 1.0, 1.5, ..., 5.5]

results = {
    "Precision": [],
    "Recall": [],
    "F1 Score": []
}



for treshold in THRESHOLDS:

    #Variables
    all_trajectory_names = list(hashed_similarities.keys()) # All trajectory names
    true_positives = 0
    false_positives = 0
    false_negatives = 0
    precision = 0 
    recall = 0
    f1_Score = 0

    # Loop through all trajectory names
    for trajectory in all_trajectory_names:
        
        # Pred and ground truth
        predicted_similar = find_predicted_similar_trajectories(trajectory, bucket_system)
        ground_truth = get_nearest_neighbour_under_threshold(trajectory, treshold, true_sim_matrix_df).index.to_list()
        true_positives += calculate_true_positives(predicted_similar, ground_truth)
        false_positives += calculate_false_positives(predicted_similar, ground_truth)
        false_negatives += calculate_false_negatives(predicted_similar, ground_truth)
        
    # Calculate precision and recall
    precision = compute_bucket_system_precision(true_positives, false_positives)
    recall = compute_bucket_system_recall(true_positives, false_negatives)
    f1_score = compute_bucket_system_f1_score(precision, recall)

    results["Precision"].append(round(precision, 3))
    results["Recall"].append(round(recall, 3))
    results["F1 Score"].append(round(f1_score,3))
    
# print bucket system stastistics for the bucketing parameters
print(f"Bucket system statistics with the parameters used to hash trajectories into buckets for city: {CITY}, measure: {MEASURE}, size: {DATA_SIZE}, diameter_bucketing: {DIAMETER_BUCKETING}, layers_bucketing: {LAYERS_BUCKETING}, disks_bucketing: {DISKS_BUCKETING}")
# Create DataFrame with thresholds as columns and metrics as row indexes
df = pd.DataFrame(results, index=[f"Threshold = {t}" for t in THRESHOLDS]).T

df

# Cell for performing several executions with different parameter values, writes all results to csv

In [ ]:

from re import L
import numpy as np
import pandas as pd
import csv
from result_analysis.disk_correlation_bucketing import fun_wrapper_corr_bucketing


# Strategies
CITY = "rome" # "rome" or "porto"
MEASURE = "dtw" # "dtw" or "frechet"
BUCKETING_METHOD = "loose" # Bucketing method to use

# Define similarity thresholds
THRESHOLDS = np.arange(1, 2.0, 1)  # Generates [1, 2, 3, 4, 5]

# Dictionary with parameter values
parameter_Values = {
    "config1": {
        "bucketing": {
            "diameter": 1.6,
            "layers": 2,
            "disks": 50
        },
        "compression": {
            "diameter": 1.5,
            "layers": 5,
            "disks": 40
        },
        "size": 50 
    },
    "config2": {
        "bucketing": {
            "diameter": 1.7,
            "layers": 3,
            "disks": 50
        },
        "compression": {
            "diameter": 1.6,
            "layers": 7,
            "disks": 40
        },
        "size": 50
    }   
}
# Prepare CSV file for results
# csv_filename = f"../../../results_hashed/bucket_evaluation/disk_bucket_evaluation_results_diameters_{DIAMETER_VALUES}_layers_{LAYERS_VALUES}_disks_{DISKS_VALUES}_sizes_{SIZE_VALUES}.csv"
csv_filename = f"../../../results_hashed/bucket_evaluation/disk_HYBRID_bucket_evaluation_results.csv"

# Open CSV file and write headers
with open(csv_filename, mode='w', newline='') as file:
    writer = csv.writer(file)
    
    
    writer.writerow(["City", "Measure", 
                    "Diameter_bucketing", "Layers_bucketing", "Disks_bucketing",
                    "Diameter_compression", "Layers_compression", "Disks_compression",
                    "Size", 
                    "Threshold", "Precision", "Recall", "F1 Score", 
                    "Total Buckets", "Largest Bucket Size", "Smallest Bucket Size", 
                    "Buckets with >1 Trajectory", "Buckets with 1 Trajectory", 
                    "Percentage >1 Trajectory", "Percentage 1 Trajectory", 
                    "Correlation Coefficient"])
    
    # Looping over the outer dictionary
    for config_name, config_data in parameter_Values.items():
        
        
        #Getting truesim matrix
        SIZE = config_data.get("size")
        file_path = f"../../../results_true/similarity_values/{CITY}/{MEASURE}/{CITY}-{MEASURE}-{SIZE}.csv"
        true_sim_matrix_df = pd.read_csv(file_path, index_col=0)
        def convert_to_float(value):
            try:
                return float(value)
            except ValueError:
                return value
        true_sim_matrix_df = true_sim_matrix_df.map(convert_to_float)
        true_sim_matrix_df = (true_sim_matrix_df + true_sim_matrix_df.T)
        
        
        # Get the bucketing parameters
        DIAMETER_BUCKETING = config_data.get("bucketing").get("diameter")
        LAYERS_BUCKETING = config_data.get("bucketing").get("layers")
        DISKS_BUCKETING = config_data.get("bucketing").get("disks")
        
        #Get the compression parameters
        DIAMETER_COMPRESSION = config_data.get("compression").get("diameter")
        LAYERS_COMPRESSION = config_data.get("compression").get("layers")
        DISKS_COMPRESSION = config_data.get("compression").get("disks") 
        
        #Prints configuration
        print(f"Running hybrid: City={CITY},Measure={MEASURE}, Diameter_bucketing={DIAMETER_BUCKETING}, Layers_bucketing={LAYERS_BUCKETING}, Disks_bucketing={DISKS_BUCKETING}, Diameter_compression={DIAMETER_COMPRESSION}, Layers_compression={LAYERS_COMPRESSION}, Disks_compression={DISKS_COMPRESSION}, Size={SIZE}")
        
        # Generate bucket system
        hashed_similarities, bucket_system = generate_disk_hash_similarity_with_bucketing_hybrid(
            city=CITY, 
            diameter_bucketing=DIAMETER_BUCKETING, diameter_compression=DIAMETER_COMPRESSION, 
            layers_bucketing=LAYERS_BUCKETING, layers_compression=LAYERS_COMPRESSION,
            disks_bucketing=DISKS_BUCKETING, disks_compression=DISKS_COMPRESSION, 
            measure=MEASURE, size=SIZE, bucketing_method=BUCKETING_METHOD
        )
        
        # Compute correlation
        corr = fun_wrapper_corr_bucketing(hashed_similarities, true_sim_matrix_df)
        
        # Evaluate the bucket system
        bucket_evaluation = evaluate_bucket_system(bucket_system)
        bucket_eval_values = bucket_evaluation.iloc[0].tolist() 
        

        # Store results for this configuration
        results = {"Precision": [], "Recall": [], "F1 Score": []}

        # Loop through each threshold
        for threshold in THRESHOLDS:
            all_trajectory_names = list(hashed_similarities.keys())  # List of trajectories
            true_positives, false_positives, false_negatives = 0, 0, 0

            # Compute precision, recall, and F1 score
            for trajectory in all_trajectory_names:
                predicted_similar = find_predicted_similar_trajectories(trajectory, bucket_system)
                ground_truth = get_nearest_neighbour_under_threshold(trajectory, threshold, true_sim_matrix_df).index.to_list()
                true_positives += calculate_true_positives(predicted_similar, ground_truth)
                false_positives += calculate_false_positives(predicted_similar, ground_truth)
                false_negatives += calculate_false_negatives(predicted_similar, ground_truth)

            # Compute final scores
            precision = round(compute_bucket_system_precision(true_positives, false_positives), 3)
            recall = round(compute_bucket_system_recall(true_positives, false_negatives),3)
            f1_score = round(compute_bucket_system_f1_score(precision, recall),3)

            # Save to dictionary
            results["Precision"].append(precision)
            results["Recall"].append(recall)
            results["F1 Score"].append(f1_score)
            
            # Write results to CSV including
            writer.writerow([CITY, MEASURE, DIAMETER_BUCKETING, LAYERS_BUCKETING, DISKS_BUCKETING, DIAMETER_COMPRESSION, LAYERS_COMPRESSION, DISKS_COMPRESSION, SIZE, 
                             threshold, precision, recall, f1_score] + bucket_eval_values + [corr])

        
        writer.writerow([])
        
        
        # Print summary for this configuration
        print(f"Completed: City={CITY},Measure={MEASURE}, Diameter_bucketing={DIAMETER_BUCKETING}, Layers_bucketing={LAYERS_BUCKETING}, Disks_bucketing={DISKS_BUCKETING}, Diameter_compression={DIAMETER_COMPRESSION}, Layers_compression={LAYERS_COMPRESSION}, Disks_compression={DISKS_COMPRESSION}, Size={SIZE}")
        print(pd.DataFrame(results, index=[f"Threshold = {t}" for t in THRESHOLDS]).T)
        print("-" * 80)
    

print(f"All results saved to {csv_filename}")

In [ ]:
# Read newly created csv file
df = pd.read_csv(csv_filename)
df